# Daily Challenge: Precision Prompting for Control & Output Quality

**Scenario.** On the Learning & Communications team, we turn a dense paragraph from an internal remote-access security policy into an employee-facing microlearning snippet for the newsletter.

**Prompt development cycle:** design → generate → evaluate → refine.
- **Step 1:** a prompt controlling tone, format, paraphrasing and length.
- **Step 2:** evaluating the output on 6 criteria (relevance, clarity, structure, tone, length, factual accuracy).
- **Step 3:** detecting hallucinations and restricting the model to the input.
- **Step 4:** a plain-language paraphrase for a junior intern.
- **Step 5:** extracting a direct quote, and when quoting beats paraphrasing.

**Model.** Every prompt is executed on an open instruction-tuned model (`Qwen/Qwen2.5-1.5B-Instruct`, free, no API key, runs on a Colab CPU or GPU). The same prompts can be pasted into ChatGPT, Gemini or DeepSeek. Each evaluation is done **both by automatic checks** (word count, bullets, verbatim copying, new terms not in the source) **and by a human review** written in the analysis cells.

In [1]:
!pip install -q "transformers<5" accelerate

In [2]:
import os, re, time
os.environ["USE_TF"] = "0"
import torch, pandas as pd
from transformers import pipeline, set_seed
set_seed(0)
pd.set_option("display.max_colwidth", 200)

MODEL_ID = "Qwen/Qwen2.5-1.5B-Instruct"
llm = pipeline("text-generation", model=MODEL_ID, device_map="auto",
               torch_dtype=torch.float16 if torch.cuda.is_available() else torch.float32)

def run(prompt, max_new_tokens=200):
    t0 = time.time()
    out = llm([{"role": "user", "content": prompt}], max_new_tokens=max_new_tokens, do_sample=False)[0]["generated_text"][-1]["content"].strip()
    print(out); print(f"\n[{count_words(out)} words, {time.time()-t0:.0f}s]")
    return out

def count_words(text):
    return len(re.findall(r"[A-Za-z0-9][A-Za-z0-9'’-]*", text))

In [3]:
SOURCE = ("Employees must ensure that all remote access to internal systems is established via the approved secure VPN. "
          "Under no circumstances should unsecured connections or personal devices lacking endpoint protection be used "
          "to access proprietary data or sensitive communications.")
print(SOURCE, f"\n\n[{count_words(SOURCE)} words]")

Employees must ensure that all remote access to internal systems is established via the approved secure VPN. Under no circumstances should unsecured connections or personal devices lacking endpoint protection be used to access proprietary data or sensitive communications. 

[38 words]


### Automatic evaluation helpers
- **Length and structure:** the word count and the number of bullet lines.
- **Paraphrase vs. quotation:** the longest run of consecutive words copied from the source. A run of 6 or more words is treated as a direct quotation.
- **Key facts kept:** the three critical security requirements must survive the rewrite.
  1. use the approved secure VPN for remote access;
  2. no unsecured connections;
  3. no personal devices without endpoint protection (security software).
- **Possible hallucinations:** content words in the output that are neither in the source nor on a short list of expected plain-language synonyms. These are flagged for **human review**, not counted as automatic errors.

In [4]:
STOP = set("""a an the and or but if to of in on at for with by from as is are be been it its this that these those you your
we our us they them their i me my do does don't not no never always only any all every each can could should must will would
may might just so also than then there here what when where which who how into out up about via use used using make sure keep
get got let's lets it's let one ones more most other such very really""".split())

ALLOWED = set("""secure security safe safely safety vpn virtual private network approved official company work
remote remotely access accessing connect connecting connection connections log logging internal systems system tools
unsecured unsafe insecure open public personal own device devices laptop laptops phone phones computer computers
endpoint protection protected protect antivirus software employees employee team everyone staff colleagues
confidential sensitive private proprietary data information info files messages communications communication
policy rule rules required requirement need needs important remember please always never avoid allowed tip tips
reminder quick home anywhere working work-from-home wfh lacking without""".split())

def stem(w):
    return re.sub(r"(ing|ed|es|s)$", "", w)

def longest_copied_run(text, source=SOURCE):
    t = re.findall(r"[a-z0-9']+", text.lower()); s = re.findall(r"[a-z0-9']+", source.lower())
    best = 0
    for i in range(len(t)):
        for j in range(len(s)):
            k = 0
            while i + k < len(t) and j + k < len(s) and t[i + k] == s[j + k]:
                k += 1
            best = max(best, k)
    return best

def evaluate(text, max_words=75, max_bullets=None):
    words = count_words(text)
    bullets = [l for l in text.splitlines() if re.match(r"^\s*([-*•]|\d+[.)])\s+", l)]
    low = text.lower()
    facts = {
        "fact: approved secure VPN": "vpn" in low,
        "fact: no unsecured connections": bool(re.search(r"unsecure|insecure|unsafe|public wi|open wi|not secure", low)),
        "fact: no unprotected personal devices": bool(re.search(r"personal|own device|your own", low)) and
                                                bool(re.search(r"endpoint|protect|antivirus|security software", low)),
    }
    src_words = {stem(w) for w in re.findall(r"[a-z]+", SOURCE.lower())}
    new_terms = sorted({w for w in re.findall(r"[a-z][a-z-]+", low)
                        if w not in STOP and stem(w) not in src_words and w not in ALLOWED and stem(w) not in {stem(a) for a in ALLOWED}})
    run_len = longest_copied_run(text)
    res = {"words": words, f"≤ {max_words} words": words <= max_words, "bullet lines": len(bullets),
           "longest copied run (words)": run_len, "paraphrased (no ≥6-word copy)": run_len < 6,
           **facts, "new terms to review": ", ".join(new_terms) or "-"}
    if max_bullets:
        res[f"≤ {max_bullets} bullets"] = 0 < len(bullets) <= max_bullets
    return res

## Step 1: Craft a prompt that controls tone, format and length

In [5]:
prompt_v1 = f"""You are an internal communications writer at a global tech company, creating a microlearning snippet
for the employee newsletter.

Rewrite the policy text below so that every employee understands it quickly:
- Tone: friendly, clear and positive (speak to the reader as "you"), no legal jargon.
- Format: a short title line, then bullet points (3-4 bullets).
- Paraphrase the meaning in your own words; do NOT copy sentences or use direct quotations.
- Length: under 75 words in total.

Policy text:
\"\"\"{SOURCE}\"\"\""""
out_v1 = run(prompt_v1)

**Title:** Secure Access to Internal Systems

*Use only approved secure VPN for remote access.*
*Avoid using unsecured connections or personal devices without proper security.*

This ensures our confidential information remains safe from unauthorized access.

[34 words, 9s]


## Step 2: Evaluate the output

In [6]:
eval_v1 = evaluate(out_v1)
pd.Series(eval_v1, name="v1").to_frame()

,v1
words,34
≤ 75 words,True
bullet lines,0
longest copied run (words),5
paraphrased (no ≥6-word copy),True
fact: approved secure VPN,True
fact: no unsecured connections,True
fact: no unprotected personal devices,False
new terms to review,"ensures, proper, remains, title, unauthorized"


### Step 2: Evaluation of v1

| Criterion | Verdict | Evidence |
|---|---|---|
| **Relevance** | ✅ mostly | It keeps the VPN rule and the "no unsecured connections or personal devices" rule. |
| **Clarity** | ✅ | Two short sentences that are much easier to read than the original 37-word legal style. |
| **Structure** | ❌ | It asked for bullets but produced italic lines (`*...*`) and a closing paragraph: **0 real bullet points**. |
| **Tone** | ✅ / ⚠️ | Neutral and clear, but not very "friendly". It uses no "you", only imperatives. |
| **Length** | ✅ | 34 words (< 75). |
| **Paraphrasing** | ✅ | The longest copied run is 5 words ("use only approved secure VPN"…). No direct quotation. |
| **Factual accuracy** | ⚠️ | 1) **Weakened detail:** "personal devices without proper security" loses the specific requirement of **endpoint protection**. "Proper security" is vague and could be read as "a password is enough". 2) **Added content:** "This ensures our confidential information remains safe from **unauthorized access**". That rationale is not in the policy. It is harmless here, but it is exactly the kind of addition that can turn into an invented rule. |

**Conclusion.** v1 is readable and short but fails the structure criterion, dilutes a critical security detail and adds an unrequested statement, so it needs revision (Step 3).

## Step 3: Detect and mitigate hallucinations
The revised prompt **restricts the model to the input**. It names the three facts that must appear, forbids new recommendations, tools or technologies, and adds a self-check before answering.

In [7]:
prompt_v2 = f"""You are an internal communications writer creating a newsletter microlearning snippet.

TASK: Paraphrase ONLY the policy text below for all employees.

RULES:
1. Only paraphrase the content in the input text. Do not add new recommendations, steps, tools, technologies,
   examples, consequences or policy terms (for example, do not mention passwords, Wi-Fi names, IT contacts or 2FA).
2. Keep all three requirements from the text: (a) remote access only through the approved secure VPN,
   (b) never use unsecured connections, (c) never use personal devices without endpoint protection (security software)
   for company data or sensitive messages.
3. Friendly, clear tone; speak to the reader as "you"; no jargon.
4. Format: one short title line, then exactly 3 bullet points (one per requirement).
5. Do not copy phrases from the text; use your own words. Stay under 75 words in total.
6. Before answering, check every bullet against the text: if a detail is not in the text, delete it.

Policy text:
\"\"\"{SOURCE}\"\"\""""
out_v2 = run(prompt_v2)

**Title:** Secure Remote Access

- **Requirement:** Use the approved secure VPN exclusively for remote access.
- **Requirement:** Never connect using unsecured networks.
- **Requirement:** Avoid accessing company data or sensitive messages on personal devices without proper security software installed.

[36 words, 12s]


In [8]:
eval_v2 = evaluate(out_v2)
pd.DataFrame({"v1 (Step 1)": eval_v1, "v2 (input-only)": eval_v2})

,v1 (Step 1),v2 (input-only)
words,34,36
≤ 75 words,True,True
bullet lines,0,3
longest copied run (words),5,4
paraphrased (no ≥6-word copy),True,True
fact: approved secure VPN,True,True
fact: no unsecured connections,True,True
fact: no unprotected personal devices,False,True
new terms to review,"ensures, proper, remains, title, unauthorized","exclusively, installed, proper, title"


### Step 3: Hallucination check and mitigation
The new-terms check flagged for v1: *ensures, remains, **unauthorized**, proper*. A human review confirms that the added sentence ("…remains safe from unauthorized access") and the vague "proper security" do not come from the source.

**Mitigation techniques used in v2:**
1. **Input-only instruction:** "Only paraphrase the content in the input text. Do not add new recommendations, steps, tools, technologies…", with concrete examples of forbidden additions (passwords, Wi-Fi names, 2FA).
2. **An explicit list of the facts that must be kept**, (a) VPN, (b) no unsecured connections, (c) no unprotected personal devices, so that the model neither drops nor softens them.
3. **A fixed structure** (exactly 3 bullets, one per requirement), which leaves no room for extra content.
4. **A self-check instruction** ("if a detail is not in the text, delete it").

**Result:** v2 has **3 proper bullets**, 36 words, all three facts (including "security software", the plain-language version of endpoint protection), no copied phrase longer than 4 words, and no added recommendation. The remaining flagged words (*exclusively, installed, proper*) are paraphrasing choices, not new facts. One small leftover: the label "**Requirement:**" repeated on each bullet is stiff. For a newsletter, a friendlier label or none at all would read better.

## Step 4: Paraphrasing deep dive for a junior intern

In [9]:
prompt_v3 = f"""You are a friendly mentor helping a junior intern on their first week.

Explain the policy text below in plain language:
- Use no more than 4 bullet points, each a short phrase (max 12 words).
- Use simple everyday words; avoid any corporate, technical or legal jargon (if you must say "VPN", explain it in a few words).
- Keep the tone supportive and informative, as if you were helping, not warning.
- Paraphrase only what the text says: do not add new rules, tools or examples.

Policy text:
\"\"\"{SOURCE}\"\"\""""
out_v3 = run(prompt_v3)

- Use approved secure VPN for remote access.
- Never use unsecured connections.
- Don't use personal devices without protection.
- Protect proprietary data and sensitive communications.

[23 words, 8s]


In [10]:
JARGON = ["endpoint", "proprietary", "establish", "under no circumstances", "internal systems", "remote access",
          "lacking", "ensure", "sensitive communications"]
eval_v3 = evaluate(out_v3, max_bullets=4)
eval_v3["jargon terms left"] = ", ".join(j for j in JARGON if j in out_v3.lower()) or "-"
eval_v3["avg words per bullet"] = round(sum(count_words(l) for l in out_v3.splitlines() if re.match(r"^\s*([-*•]|\d+[.)])\s+", l))
                                        / max(eval_v3["bullet lines"], 1), 1)
pd.Series(eval_v3, name="v3 (intern)").to_frame()

,v3 (intern)
words,23
≤ 75 words,True
bullet lines,4
longest copied run (words),3
paraphrased (no ≥6-word copy),True
fact: approved secure VPN,True
fact: no unsecured connections,True
fact: no unprotected personal devices,True
new terms to review,don
≤ 4 bullets,True


### Step 4: Intern version (v3)
- ✅ **4 bullets** (the maximum), very short phrases (5.8 words per bullet on average), supportive and neutral tone, all three facts kept, and no new rule.
- ❌ **Jargon left:** *proprietary data*, *sensitive communications*, *remote access* survive. "Proprietary" in particular is not plain language for an intern.
- ⚠️ The 4th bullet ("Protect proprietary data and sensitive communications") restates the goal rather than giving an action.

Next we refine the prompt by listing the jargon to avoid and suggesting plain alternatives.

In [11]:
prompt_v3b = f"""You are a friendly mentor helping a junior intern on their first week.
Explain the policy text below in plain, everyday language.

Rules:
- Exactly 3 bullet points, each a short phrase of max 12 words. No title, no extra sentences.
- Do NOT use these jargon words: "proprietary", "endpoint", "remote access", "internal systems",
  "sensitive communications", "unsecured", "ensure". Instead say things like "the company's secure connection (VPN)",
  "public or unprotected Wi-Fi", "your own laptop or phone without security software", "company files and private messages".
- Supportive tone: help, don't warn. Use "you".
- Keep exactly the meaning of the text; add nothing new.

Policy text:
\"\"\"{SOURCE}\"\"\""""
out_v3b = run(prompt_v3b)
eval_v3b = evaluate(out_v3b, max_bullets=4)
eval_v3b["jargon terms left"] = ", ".join(j for j in JARGON if j in out_v3b.lower()) or "-"
pd.DataFrame({"v3 (intern)": eval_v3, "v3b (refined)": eval_v3b})

Remote access must go through approved secure VPN.
Use only secured connections.
No personal devices allowed for accessing data.

[19 words, 6s]


,v3 (intern),v3b (refined)
words,23,19
≤ 75 words,True,True
bullet lines,4,0
longest copied run (words),3,3
paraphrased (no ≥6-word copy),True,True
fact: approved secure VPN,True,True
fact: no unsecured connections,True,False
fact: no unprotected personal devices,True,False
new terms to review,don,"go, secured, through"
≤ 4 bullets,True,False


### Step 4 refinement (v3b): a lesson in over-correction
The refined prompt removed most jargon, but the output **got worse**:
- **A factual distortion:** "No personal devices allowed for accessing data". The policy only forbids personal devices **lacking endpoint protection**; it does not ban personal devices. Simplifying too hard dropped the condition and **invented a stricter rule**, the most dangerous kind of error in a security communication: employees would be misinformed, or IT would get unnecessary tickets.
- **Format lost:** plain lines instead of bullet points, despite "exactly 3 bullet points".
- "Remote access" is still present.

**Takeaways from the refinement cycle:**
- A small model cannot satisfy many simultaneous constraints (no jargon + exact meaning + format), and **constraints compete**: pushing hard on "simple words" sacrificed accuracy.
- Always re-evaluate after each refinement, since a change can introduce new errors.
- The best version for the intern is **v3**, with the 4th bullet rewritten by a human ("Keep company files and private messages safe"). A stronger model (GPT-4-class) would usually handle v3b correctly. Even so, a **human review against the source** is mandatory for policy communications.

## Step 5: Quote extraction variant

In [12]:
prompt_quote = f"""From the policy text below, extract ONE direct quote that best captures the core security rule.

Rules:
- Copy the words exactly as written, character for character, inside double quotation marks.
- Do not shorten, rephrase, merge sentences or add words.
- On the next line, write one sentence (max 20 words) explaining why you chose this quote.

Policy text:
\"\"\"{SOURCE}\"\"\""""
out_quote = run(prompt_quote, max_new_tokens=120)

**Direct Quote:**
"Employees must ensure that all remote access to internal systems is established via the approved secure VPN."

**Explanation:**
I chose this quote because it directly addresses the core security rule of using an approved secure VPN for remote access to internal systems. It emphasizes the importance of securing remote connections and avoiding insecure methods like unsecured connections or personal devices without endpoint protection. This rule ensures that only authorized and protected channels are used to access company data and sensitive communications.

[83 words, 19s]


In [13]:
quotes = re.findall(r"[\"“]([^\"”]{10,})[\"”]", out_quote)
norm = lambda s: re.sub(r"\s+", " ", s.strip().rstrip(".").lower())
for q in quotes:
    print(f"Quote: {q!r}\n  verbatim in source: {norm(q) in norm(SOURCE)} | {count_words(q)} words")
if not quotes:
    print("No quoted text found in the output.")

Quote: 'Employees must ensure that all remote access to internal systems is established via the approved secure VPN.'
  verbatim in source: True | 17 words


### Step 5: Quote extraction
✅ The model extracted a **verbatim** quote (checked automatically: it is an exact substring of the source):

> "Employees must ensure that all remote access to internal systems is established via the approved secure VPN."

It is a good choice because it states the positive rule, the one action everyone must take. The second sentence (the prohibitions) would be the alternative choice. ❌ The explanation ignored the length limit: 83 words instead of 20 or fewer. The quote constraint was respected; the secondary constraint was not.

**When is quoting more appropriate than paraphrasing?**
- **Compliance, legal and HR communications**, where the exact wording is binding: policy acknowledgements, audit evidence, security incident notices, disciplinary or contractual matters, regulatory statements (GDPR, SOC 2, ISO 27001).
- **Official announcements** from leadership or the CISO, where attributing the exact words matters, and **training or quizzes** that must test knowledge of the precise rule ("Under no circumstances…").
- Any time a paraphrase could change the scope of an obligation. Step 4b showed how easily "personal devices *lacking endpoint protection*" becomes "no personal devices".

**When might quoting pose a risk?**
- **Out of context:** a single quoted sentence can mislead. Quoting only the VPN sentence omits the prohibition on unprotected personal devices, so readers may think the VPN alone is enough.
- **Comprehension:** dense legal jargon ("endpoint protection", "proprietary data") may not be understood by non-technical staff, so the message is technically accurate but not followed.
- **Accuracy and freshness:** a model may produce a "quote" that is slightly altered or invented. Verify it verbatim, as done above. A quote also becomes wrong when the policy is updated.
- **Confidentiality:** quoting internal policy text in widely distributed or external channels may expose security details or confidential wording.

**Best practice:** combine both. Use a short verbatim quote for the binding rule, followed by a plain-language paraphrase of what it means in practice, and link to the full policy.